# Somneiros — Production Dream Interpretation Database Builder

This interactive Google Colab notebook builds the **Somneiros Dream Interpretation Database (DID)** as a versioned, citation-backed research corpus.

It is designed for a full production run rather than a small demonstration. The default taxonomy contains hundreds of dream symbols, actions, emotions, people, animals, body experiences, settings, objects, natural events, life situations, and dream phenomena.

## Non-negotiable evidence rules

- Contemporary sleep science, historical interpretation, religious tradition, and reflective symbolism remain separate.
- No universal symbol meaning is presented as scientifically proven.
- A source must exist, load successfully, and support the associated claim.
- Disputed stories are labeled as disputed.
- Cultural material must identify a specific community, text, author, place, or period.
- Records containing invented citations, diagnosis, prophecy, recovered-memory claims, or supernatural certainty are rejected.
- The app-facing database uses cautious language: *may*, *can invite reflection*, and *depends on personal context*.

## Production outputs

The notebook exports:

- `did_records.json` — complete research records
- `did_sources.json` — normalized source registry
- `did_relationships.json` — related concepts and graph edges
- `did_taxonomy.json` — full term and category index
- `did_app.json` — compact PWA database consumed by `app.js`
- `did_records.csv` — spreadsheet-friendly record summary
- `did_sources.csv` — spreadsheet-friendly source inventory
- `did_qa_report.csv` — acceptance, rejection, and quality metrics
- `did_rejected.json` — records requiring review
- `somneiros_did.sqlite` — relational SQLite database
- `somneiros_did_bundle.zip` — upload-ready bundle

## 1. Install the research stack

The notebook uses public scholarly APIs for metadata and literature discovery, plus the OpenAI Responses API for web-assisted source discovery, structured extraction, synthesis, and quality review.

In [ ]:
!pip -q install -U openai pydantic pandas requests ipywidgets nltk

In [ ]:
from __future__ import annotations
import os, re, json, time, math, hashlib, sqlite3, zipfile, shutil, threading
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Optional, Literal
from urllib.parse import urlparse, quote
from collections import defaultdict

import requests
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

OUTPUT_DIR = Path('/content/somneiros_did')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
RAW_DIR = OUTPUT_DIR / 'raw'
for path in (OUTPUT_DIR, CHECKPOINT_DIR, RAW_DIR): path.mkdir(parents=True, exist_ok=True)



def retrying(max_attempts: int = 3, base_delay: float = 1.0):
    def decorator(func):
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as error:
                    last_error = error
                    if attempt + 1 < max_attempts:
                        time.sleep(base_delay * (2 ** attempt))
            raise last_error
        return wrapper
    return decorator

RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()
print('Output directory:', OUTPUT_DIR)

## 2. Connect API keys securely

In Google Colab, open the **Secrets** panel and add:

- `OPENAI_API_KEY` — required for agentic web research and record synthesis
- `NCBI_API_KEY` — optional; raises the NCBI request-rate allowance
- `NCBI_EMAIL` — recommended by NCBI for E-utilities requests
- `OPENALEX_API_KEY` — optional depending on current OpenAlex access requirements

Keys are read from Colab Secrets or environment variables and are never written into the exported DID files.

In [ ]:
def read_secret(name: str, default: str = '') -> str:
    value = os.getenv(name, '')
    if value: return value
    try:
        from google.colab import userdata
        return userdata.get(name) or default
    except Exception:
        return default

OPENAI_API_KEY = read_secret('OPENAI_API_KEY')
NCBI_API_KEY = read_secret('NCBI_API_KEY')
NCBI_EMAIL = read_secret('NCBI_EMAIL', 'replace-with-your-email@example.com')
OPENALEX_API_KEY = read_secret('OPENALEX_API_KEY')

if not OPENAI_API_KEY:
    print('OPENAI_API_KEY is not set. Add it in Colab Secrets before running the build.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    print('OpenAI client ready.')

## 3. Interactive production settings

**Full Production** is the default. The controls allow a smaller validation run without changing the notebook code. The minimum-source setting applies to source-derived claims; reflective-only records are permitted only at low confidence and without pretending to be scientific evidence.

In [ ]:
MODE = widgets.Dropdown(options=['Full Production','Validation Run','Custom'], value='Full Production', description='Run mode:')
RESEARCH_MODEL = widgets.Dropdown(options=['gpt-5-mini','gpt-5','gpt-4.1-mini'], value='gpt-5-mini', description='Research:')
SYNTHESIS_MODEL = widgets.Dropdown(options=['gpt-5','gpt-5-mini','gpt-4.1'], value='gpt-5', description='Synthesis:')
MAX_RECORDS = widgets.IntText(value=0, description='Max records:', tooltip='0 means every taxonomy term')
MIN_SOURCES = widgets.IntSlider(value=3, min=1, max=8, step=1, description='Min sources:')
MAX_SOURCES = widgets.IntSlider(value=10, min=3, max=20, step=1, description='Max sources:')
MIN_QA_SCORE = widgets.FloatSlider(value=0.72, min=0.5, max=0.95, step=0.01, description='QA threshold:')
USE_WEB = widgets.Checkbox(value=True, description='OpenAI web search')
USE_PUBMED = widgets.Checkbox(value=True, description='PubMed / PMC')
USE_OPENALEX = widgets.Checkbox(value=True, description='OpenAlex')
USE_CROSSREF = widgets.Checkbox(value=True, description='Crossref')
USE_EMBEDDINGS = widgets.Checkbox(value=False, description='Semantic relationship pass')
RESUME = widgets.Checkbox(value=True, description='Resume checkpoints')

settings_box = widgets.VBox([
    MODE, widgets.HBox([RESEARCH_MODEL,SYNTHESIS_MODEL]), widgets.HBox([MAX_RECORDS,MIN_SOURCES,MAX_SOURCES]),
    MIN_QA_SCORE, widgets.HBox([USE_WEB,USE_PUBMED,USE_OPENALEX,USE_CROSSREF]),
    widgets.HBox([USE_EMBEDDINGS,RESUME])
])
display(settings_box)

def resolved_settings():
    max_records = MAX_RECORDS.value
    if MODE.value == 'Validation Run' and max_records == 0: max_records = 12
    return {
        'mode': MODE.value, 'research_model': RESEARCH_MODEL.value, 'synthesis_model': SYNTHESIS_MODEL.value,
        'max_records': max_records, 'min_sources': MIN_SOURCES.value, 'max_sources': MAX_SOURCES.value,
        'min_qa_score': MIN_QA_SCORE.value,
        'use_web': USE_WEB.value, 'use_pubmed': USE_PUBMED.value, 'use_openalex': USE_OPENALEX.value,
        'use_crossref': USE_CROSSREF.value, 'use_embeddings': USE_EMBEDDINGS.value, 'resume': RESUME.value
    }
resolved_settings()

## 4. Production taxonomy

In [ ]:
SEED_TAXONOMY = {'dream_experiences': ['bad dream', 'nightmare', 'recurring dream', 'lucid dream', 'false awakening', 'sleep paralysis', 'vivid dream', 'fragmented dream', 'dream within a dream', 'shared dream belief', 'prophetic dream belief', 'visitation dream belief', 'out-of-body sensation', 'déjà vu in a dream', 'forgotten dream', 'waking from a dream', 'dreaming in black and white', 'dreaming in color', 'hearing music in a dream', 'unable to wake up'], 'actions_situations': ['being chased', 'chasing someone', 'hiding', 'searching', 'being lost', 'getting trapped', 'escaping', 'running', 'unable to run', 'walking', 'climbing', 'falling', 'flying', 'floating', 'swimming', 'drowning', 'driving', 'losing control of a vehicle', 'missing transportation', 'being late', 'taking a test', 'failing a test', 'graduating', 'working', 'being fired', 'winning', 'losing', 'fighting', 'arguing', 'being attacked', 'attacking someone', 'killing', 'being killed', 'rescuing someone', 'being rescued', 'stealing', 'being robbed', 'breaking into a house', 'being watched', 'being followed', 'performing on stage', 'speaking in public', 'unable to speak', 'screaming without sound', 'singing', 'dancing', 'laughing', 'crying', 'kissing', 'having sex', 'getting married', 'attending a wedding', 'attending a funeral', 'giving birth', 'being pregnant', 'packing', 'unpacking', 'moving house', 'traveling', 'getting lost while traveling', 'waiting', 'opening a door', 'locking a door', 'finding something', 'losing something', 'cleaning', 'cooking', 'eating', 'drinking', 'shopping', 'building', 'destroying', 'digging', 'planting', 'harvesting', 'reading', 'writing', 'taking a photograph', 'using a phone', 'sending a message', 'receiving a message', 'time travel', 'apocalypse', 'war', 'explosion', 'natural disaster', 'celebration', 'reunion', 'abandonment', 'betrayal', 'confession', 'forgiveness'], 'emotions_states': ['fear', 'anxiety', 'panic', 'joy', 'happiness', 'sadness', 'grief', 'anger', 'rage', 'shame', 'guilt', 'embarrassment', 'jealousy', 'envy', 'loneliness', 'relief', 'peace', 'confusion', 'awe', 'wonder', 'disgust', 'love', 'desire', 'hope', 'helplessness', 'confidence', 'curiosity', 'nostalgia', 'frustration', 'excitement', 'calm', 'numbness', 'urgency', 'safety', 'danger', 'freedom', 'confinement'], 'people_relationships': ['mother', 'father', 'parent', 'child', 'baby', 'son', 'daughter', 'brother', 'sister', 'grandmother', 'grandfather', 'grandparent', 'spouse', 'husband', 'wife', 'partner', 'ex-partner', 'friend', 'best friend', 'enemy', 'rival', 'stranger', 'crowd', 'neighbor', 'coworker', 'boss', 'teacher', 'student', 'doctor', 'nurse', 'therapist', 'police officer', 'firefighter', 'soldier', 'priest', 'pastor', 'rabbi', 'imam', 'religious leader', 'celebrity', 'politician', 'king', 'queen', 'bride', 'groom', 'deceased loved one', 'ancestor', 'ghost', 'double or doppelgänger', 'younger self', 'older self', 'unknown child', 'masked person', 'faceless person', 'intruder', 'criminal', 'hero', 'villain', 'wise elder', 'trickster', 'caregiver'], 'animals_creatures': ['dog', 'puppy', 'cat', 'kitten', 'horse', 'cow', 'bull', 'pig', 'goat', 'sheep', 'rabbit', 'deer', 'moose', 'bear', 'wolf', 'fox', 'coyote', 'lion', 'tiger', 'leopard', 'elephant', 'giraffe', 'monkey', 'ape', 'gorilla', 'rat', 'mouse', 'bat', 'snake', 'lizard', 'crocodile', 'alligator', 'turtle', 'frog', 'spider', 'scorpion', 'bee', 'wasp', 'ant', 'cockroach', 'beetle', 'butterfly', 'moth', 'fly', 'mosquito', 'worm', 'bird', 'eagle', 'hawk', 'owl', 'crow', 'raven', 'dove', 'chicken', 'duck', 'goose', 'swan', 'peacock', 'fish', 'shark', 'whale', 'dolphin', 'octopus', 'jellyfish', 'crab', 'lobster', 'dragon', 'dinosaur', 'unicorn', 'monster', 'alien', 'pet', 'wild animal', 'talking animal', 'injured animal', 'dead animal', 'animal attack'], 'body_health': ['body', 'face', 'eyes', 'blindness', 'ears', 'hearing loss', 'mouth', 'tongue', 'teeth', 'teeth falling out', 'hair', 'hair loss', 'hands', 'fingers', 'arms', 'legs', 'feet', 'heart', 'skin', 'blood', 'bones', 'wound', 'scar', 'illness', 'disease', 'fever', 'pain', 'injury', 'hospitalization', 'surgery', 'paralysis', 'unable to breathe', 'choking', 'vomiting', 'urination', 'defecation', 'nakedness', 'clothing malfunction', 'aging', 'becoming young', 'death', 'dying', 'dead body', 'birth', 'pregnancy', 'miscarriage', 'menstruation', 'weight change', 'transformation of the body', 'missing body part', 'extra limb', 'superhuman strength'], 'places_settings': ['house', 'childhood home', 'unknown house', 'mansion', 'abandoned house', 'apartment', 'bedroom', 'bathroom', 'kitchen', 'living room', 'hallway', 'basement', 'attic', 'closet', 'garage', 'garden', 'backyard', 'school', 'classroom', 'workplace', 'office', 'factory', 'hospital', 'church', 'temple', 'mosque', 'synagogue', 'cemetery', 'funeral home', 'prison', 'courtroom', 'police station', 'hotel', 'restaurant', 'store', 'shopping mall', 'theater', 'stadium', 'airport', 'train station', 'bus station', 'parking lot', 'road', 'highway', 'bridge', 'tunnel', 'elevator', 'stairs', 'maze', 'forest', 'jungle', 'field', 'farm', 'mountain', 'cliff', 'cave', 'desert', 'island', 'beach', 'ocean', 'river', 'lake', 'waterfall', 'swamp', 'city', 'small town', 'foreign country', 'outer space', 'moon', 'underground place', 'ruins', 'castle', 'battlefield', 'amusement park', 'playground', 'library', 'museum', 'ship', 'airplane cabin', 'vehicle interior', 'home that keeps changing', 'endless room'], 'transportation': ['car', 'truck', 'bus', 'train', 'subway', 'airplane', 'helicopter', 'boat', 'ship', 'canoe', 'kayak', 'bicycle', 'motorcycle', 'scooter', 'horseback riding', 'spaceship', 'ambulance', 'police car', 'taxi', 'ride share', 'vehicle crash', 'plane crash', 'train derailment', 'sinking ship', 'flat tire', 'broken vehicle', 'lost keys', 'wrong-way driving', 'back-seat driving', 'driverless vehicle'], 'objects_technology': ['door', 'window', 'mirror', 'clock', 'watch', 'key', 'lock', 'phone', 'computer', 'television', 'camera', 'photograph', 'book', 'letter', 'journal', 'map', 'compass', 'suitcase', 'bag', 'box', 'gift', 'money', 'coins', 'credit card', 'jewelry', 'ring', 'necklace', 'clothes', 'uniform', 'shoes', 'hat', 'mask', 'glasses', 'weapon', 'gun', 'knife', 'sword', 'shield', 'rope', 'chain', 'ladder', 'chair', 'bed', 'table', 'food', 'bread', 'fruit', 'meat', 'cake', 'water bottle', 'medicine', 'candle', 'lamp', 'flashlight', 'fire alarm', 'musical instrument', 'toy', 'doll', 'statue', 'painting', 'computer game', 'robot', 'artificial intelligence', 'social media', 'email', 'text message', 'password', 'broken screen', 'lost phone', 'unknown machine'], 'nature_weather': ['water', 'rain', 'storm', 'thunder', 'lightning', 'tornado', 'hurricane', 'flood', 'tsunami', 'snow', 'ice', 'hail', 'fog', 'wind', 'fire', 'wildfire', 'smoke', 'earthquake', 'volcano', 'avalanche', 'mud', 'sand', 'sun', 'moon', 'stars', 'eclipse', 'rainbow', 'darkness', 'light', 'shadow', 'tree', 'forest fire', 'flower', 'rose', 'garden', 'grass', 'seed', 'fruit tree', 'mountain', 'rock', 'crystal', 'ocean wave', 'river current', 'deep water', 'clear water', 'dirty water', 'underwater', 'sky', 'cloud', 'sunrise', 'sunset', 'night', 'daytime', 'season change'], 'colors_numbers_shapes': ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white', 'gray', 'gold', 'silver', 'number zero', 'number one', 'number two', 'number three', 'number four', 'number five', 'number six', 'number seven', 'number eight', 'number nine', 'number ten', 'number eleven', 'number twelve', 'number thirteen', 'repeating numbers', 'circle', 'triangle', 'square', 'spiral', 'cross', 'star', 'heart shape', 'maze pattern'], 'life_events_roles': ['new job', 'job loss', 'retirement', 'promotion', 'school enrollment', 'graduation', 'marriage', 'divorce', 'breakup', 'new relationship', 'moving', 'new home', 'travel', 'military service', 'deployment', 'homecoming', 'illness in family', 'caregiving', 'financial stress', 'inheritance', 'public recognition', 'failure', 'success', 'competition', 'birthday', 'holiday', 'family gathering', 'childhood memory', 'traumatic memory', 'unfinished task', 'major decision', 'secret', 'identity change', 'religious conversion', 'creative breakthrough', 'scientific discovery'], 'spiritual_mythic': ['angel', 'demon', 'devil', 'god or deity', 'religious figure', 'heaven', 'hell', 'afterlife', 'reincarnation', 'prayer', 'ritual', 'sacred object', 'cross symbol', 'crescent symbol', 'mandala', 'oracle', 'prophecy belief', 'miracle belief', 'curse belief', 'possession belief', 'guardian figure', 'spirit guide belief', 'ancestor spirit belief', 'mythical journey', 'underworld', 'resurrection', 'sacrifice', 'temptation', 'judgment', 'sin', 'redemption', 'blessing', 'pilgrimage', 'holy water', 'sacred mountain']}

CATEGORY_DESCRIPTIONS = {
    'dream_experiences':'Dream-state experiences and awareness phenomena',
    'actions_situations':'Actions, conflicts, tasks, and recurring situations',
    'emotions_states':'Emotional states and felt qualities',
    'people_relationships':'People, social roles, and relationships',
    'animals_creatures':'Animals, insects, marine life, and imagined creatures',
    'body_health':'Body, health, vulnerability, and physical sensations',
    'places_settings':'Buildings, rooms, landscapes, and social settings',
    'transportation':'Vehicles, travel, direction, and movement',
    'objects_technology':'Objects, tools, media, and modern technology',
    'nature_weather':'Weather, elements, landscapes, plants, and celestial imagery',
    'colors_numbers_shapes':'Colors, numbers, shapes, and repeating patterns',
    'life_events_roles':'Transitions, responsibilities, milestones, and memories',
    'spiritual_mythic':'Religious, spiritual, mythic, and moral imagery'
}
ALIASES = {'being chased': ['chased', 'someone chasing me', 'pursued'], 'teeth falling out': ['losing teeth', 'teeth breaking', 'teeth crumbling'], 'unable to run': ['running slowly', 'legs will not move'], 'unable to speak': ['no voice', 'cannot talk'], 'screaming without sound': ['silent scream'], 'deceased loved one': ['dead relative', 'dead friend'], 'double or doppelgänger': ['double', 'look-alike'], 'sleep paralysis': ['cannot move while waking'], 'false awakening': ['thought I woke up'], 'lucid dream': ['aware I was dreaming'], 'prophetic dream belief': ['precognitive dream', 'dream predicting future'], 'visitation dream belief': ['visitation dream'], 'water': ['ocean water', 'river water'], 'house': ['home'], 'car': ['automobile'], 'airplane': ['plane'], 'phone': ['cell phone', 'mobile phone'], 'money': ['cash'], 'police officer': ['police', 'cop'], 'death': ['dead', 'dying'], 'baby': ['infant'], 'child': ['kid'], 'mother': ['mom'], 'father': ['dad'], 'grandmother': ['grandma'], 'grandfather': ['grandpa'], 'spouse': ['husband', 'wife'], 'dog': ['puppy'], 'cat': ['kitten'], 'bird': ['birds'], 'snake': ['serpent'], 'spider': ['spiders'], 'ocean': ['sea'], 'river': ['stream'], 'storm': ['thunderstorm'], 'tornado': ['twister'], 'fire': ['flames'], 'forest': ['woods'], 'cemetery': ['graveyard'], 'workplace': ['job', 'office'], 'school': ['classroom'], 'bathroom': ['restroom'], 'elevator': ['lift'], 'stairs': ['staircase'], 'gun': ['firearm'], 'knife': ['blade'], 'mirror': ['reflection'], 'clock': ['timepiece'], 'key': ['keys'], 'wedding': ['marriage ceremony'], 'funeral': ['burial'], 'pregnancy': ['pregnant'], 'birth': ['giving birth']}

def slugify(text: str) -> str:
    return re.sub(r'[^a-z0-9]+','-',text.lower()).strip('-')

TAXONOMY_ROWS = []
for category, terms in SEED_TAXONOMY.items():
    for term in terms:
        TAXONOMY_ROWS.append({'term_id':slugify(term),'term':term,'display_name':term.title(),'category':category,'aliases':ALIASES.get(term,[])})

# Deduplicate while keeping first category assignment.
seen=set(); TAXONOMY_ROWS=[row for row in TAXONOMY_ROWS if not (row['term_id'] in seen or seen.add(row['term_id']))]
print(f'{len(TAXONOMY_ROWS):,} unique DID concepts across {len(SEED_TAXONOMY)} categories.')
pd.DataFrame(TAXONOMY_ROWS).groupby('category').size().sort_values(ascending=False).to_frame('terms')

## 5. Data schema

A source record stores provenance. An evidence claim links a precise statement to one or more sources. A DID record stores distinct interpretive perspectives, confidence, cautions, personal reflection prompts, and QA results.

In [ ]:
class SourceRecord(BaseModel):
    source_id: str
    title: str
    url: str
    source_type: Literal['peer_reviewed','book','primary_historical','museum_archive','government','university','religious_text','reference','news_interview','other']
    publisher_or_journal: str = ''
    authors: list[str] = Field(default_factory=list)
    year: Optional[int] = None
    doi: Optional[str] = None
    pmid: Optional[str] = None
    accessed_at: str
    abstract_or_excerpt: str = ''
    tradition_or_context: str = ''
    reliability_notes: str = ''
    url_verified: bool = False
    quality_score: float = Field(ge=0, le=1)

class EvidenceClaim(BaseModel):
    claim: str
    claim_type: Literal['sleep_science','historical_account','cultural_tradition','psychological_theory','biographical_account','reflective_synthesis']
    source_ids: list[str] = Field(default_factory=list)
    support_level: Literal['direct','indirect','contextual','disputed']
    caveat: str = ''

class InterpretationPerspective(BaseModel):
    perspective: str
    summary: str
    themes: list[str] = Field(default_factory=list)
    source_ids: list[str] = Field(default_factory=list)
    evidence_strength: Literal['reflective','limited','moderate','strong','historical','disputed']
    cautions: list[str] = Field(default_factory=list)

class DIDRecord(BaseModel):
    record_id: str
    term: str
    display_name: str
    category: str
    aliases: list[str] = Field(default_factory=list)
    plain_definition: str
    common_contexts: list[str] = Field(default_factory=list)
    interpretations: list[InterpretationPerspective]
    evidence_claims: list[EvidenceClaim] = Field(default_factory=list)
    overall_summary: str
    themes: list[str]
    reflection_prompts: list[str]
    related_terms: list[str] = Field(default_factory=list)
    source_ids: list[str] = Field(default_factory=list)
    evidence_label: str
    confidence_score: float = Field(ge=0, le=1)
    safety_notes: list[str]
    cultural_context_notes: list[str] = Field(default_factory=list)
    reviewed_at: str
    version: str = '1.0'

class QAResult(BaseModel):
    record_id: str
    accepted: bool
    score: float = Field(ge=0, le=1)
    source_score: float = Field(ge=0, le=1)
    provenance_score: float = Field(ge=0, le=1)
    cultural_context_score: float = Field(ge=0, le=1)
    safety_score: float = Field(ge=0, le=1)
    clarity_score: float = Field(ge=0, le=1)
    problems: list[str] = Field(default_factory=list)
    required_changes: list[str] = Field(default_factory=list)

## 6. Source discovery connectors

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({'User-Agent':f'SomneirosDID/1.0 ({NCBI_EMAIL})'})

TRUSTED_DOMAIN_WEIGHTS = {
    'nih.gov':1.0,'ncbi.nlm.nih.gov':1.0,'nobelprize.org':0.98,'loc.gov':0.96,
    'si.edu':0.95,'metmuseum.org':0.94,'britishmuseum.org':0.94,'edu':0.9,
    'openalex.org':0.88,'crossref.org':0.88,'gutenberg.org':0.82,'archive.org':0.78,
    'wikisource.org':0.72,'wikipedia.org':0.55
}

def now_iso(): return datetime.now(timezone.utc).isoformat()
def source_id(url: str, title: str='') -> str: return hashlib.sha256((url.strip()+title.strip()).encode()).hexdigest()[:16]
def domain_weight(url: str) -> float:
    host=urlparse(url).netloc.lower()
    for domain,score in TRUSTED_DOMAIN_WEIGHTS.items():
        if host==domain or host.endswith('.'+domain): return score
    return 0.5

@retrying(max_attempts=3, base_delay=1.0)
def get_json(url: str, params: Optional[dict]=None, timeout: int=30):
    response=SESSION.get(url,params=params,timeout=timeout); response.raise_for_status(); return response.json()

@retrying(max_attempts=3, base_delay=1.0)
def verify_url(url: str) -> tuple[bool,str]:
    try:
        response=SESSION.get(url,timeout=20,allow_redirects=True,stream=True)
        ok=response.status_code<400
        return ok,response.url
    except Exception:
        return False,url

def inverted_abstract(index: Optional[dict]) -> str:
    if not index: return ''
    positions=[]
    for word,locs in index.items():
        for loc in locs: positions.append((loc,word))
    return ' '.join(word for _,word in sorted(positions))

def search_openalex(query: str, limit: int=8) -> list[dict]:
    params={'search':query,'per_page':limit,'select':'id,doi,title,display_name,publication_year,authorships,primary_location,abstract_inverted_index,cited_by_count,is_retracted'}
    if OPENALEX_API_KEY: params['api_key']=OPENALEX_API_KEY
    data=get_json('https://api.openalex.org/works',params=params)
    results=[]
    for item in data.get('results',[]):
        if item.get('is_retracted'): continue
        loc=item.get('primary_location') or {}; source=(loc.get('source') or {})
        doi=item.get('doi') or ''
        url=doi or loc.get('landing_page_url') or item.get('id','')
        results.append({'title':item.get('display_name') or item.get('title',''),'url':url,'year':item.get('publication_year'),'doi':doi.replace('https://doi.org/',''),'authors':[a.get('author',{}).get('display_name','') for a in item.get('authorships',[])[:12]],'publisher_or_journal':source.get('display_name',''),'abstract_or_excerpt':inverted_abstract(item.get('abstract_inverted_index'))[:5000],'source_type':'peer_reviewed','quality_hint':min(1,0.65+math.log10(1+item.get('cited_by_count',0))/10)})
    return results

def search_crossref(query: str, limit: int=8) -> list[dict]:
    data=get_json('https://api.crossref.org/works',params={'query':query,'rows':limit,'select':'DOI,title,author,published,container-title,URL,type,is-referenced-by-count'})
    results=[]
    for item in data.get('message',{}).get('items',[]):
        title=(item.get('title') or [''])[0]; container=(item.get('container-title') or [''])[0]
        parts=((item.get('published') or {}).get('date-parts') or [[None]])[0]
        results.append({'title':title,'url':item.get('URL') or ('https://doi.org/'+item.get('DOI','')),'year':parts[0] if parts else None,'doi':item.get('DOI'),'authors':[' '.join(filter(None,[a.get('given'),a.get('family')])) for a in item.get('author',[])[:12]],'publisher_or_journal':container,'abstract_or_excerpt':'','source_type':'peer_reviewed','quality_hint':min(1,0.62+math.log10(1+item.get('is-referenced-by-count',0))/12)})
    return results

def search_pubmed(query: str, limit: int=8) -> list[dict]:
    base='https://eutils.ncbi.nlm.nih.gov/entrez/eutils/'
    params={'db':'pubmed','term':query,'retmode':'json','retmax':limit,'tool':'SomneirosDID','email':NCBI_EMAIL}
    if NCBI_API_KEY: params['api_key']=NCBI_API_KEY
    ids=get_json(base+'esearch.fcgi',params=params).get('esearchresult',{}).get('idlist',[])
    if not ids: return []
    sparams={'db':'pubmed','id':','.join(ids),'retmode':'json','version':'2.0','tool':'SomneirosDID','email':NCBI_EMAIL}
    if NCBI_API_KEY: sparams['api_key']=NCBI_API_KEY
    summary=get_json(base+'esummary.fcgi',params=sparams).get('result',{})
    results=[]
    for pmid in ids:
        item=summary.get(pmid,{})
        year=None
        match=re.search(r'(19|20)\d{2}',item.get('pubdate',''))
        if match: year=int(match.group())
        results.append({'title':item.get('title',''),'url':f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/','year':year,'pmid':pmid,'authors':[a.get('name','') for a in item.get('authors',[])[:12]],'publisher_or_journal':item.get('fulljournalname',''),'abstract_or_excerpt':'','source_type':'peer_reviewed','quality_hint':0.9})
    return results

## 7. Agent prompts and structured-output helper

The web scout discovers sources. A second structured pass extracts only claims that can be supported by those sources. The synthesizer builds the final record, and a separate reviewer attempts to reject it.

In [ ]:
WEB_SCOUT_INSTRUCTIONS = 'You are the Somneiros source-discovery agent. Find real, accessible, high-quality sources relevant to a dream concept. Separate: (1) contemporary dream/sleep science, (2) documented historical or religious interpretation traditions, (3) first-person biographical dream accounts, and (4) reflective symbolism. Never invent a title, URL, quotation, author, DOI, or cultural tradition. Prefer peer-reviewed articles, government or university sources, museums, archives, public-domain primary texts, and direct interviews. Clearly label disputed anecdotes and say when no reliable tradition-specific source is found.'

SYNTHESIS_INSTRUCTIONS = 'You build records for the Somneiros Dream Interpretation Database. Use only the supplied source packet. Do not create a universal meaning. Keep scientific findings separate from historical theories, cultural traditions, and reflective synthesis. Avoid diagnosis, prophecy, supernatural certainty, recovered-memory claims, and claims that a symbol predicts death or future events. Cultural claims must remain attached to a specific text, author, community, place, and period. Use plain language. Every source-derived interpretation or claim must cite source IDs from the supplied packet. If the packet does not support a perspective, omit it.'

QA_INSTRUCTIONS = 'Act as a hostile evidence reviewer for the Somneiros DID. Reject invented citations, claims not supported by source IDs, false universality, flattened cultural traditions, diagnosis, prophecy, recovered-memory claims, or overconfident wording. Check clarity for a general audience. Return a strict quality assessment.'

@retrying(max_attempts=3, base_delay=1.0)
def web_scout(term: str, category: str, model: str) -> tuple[str,list[dict]]:
    prompt=f"Research the dream concept: {term!r}. Category: {category}. Find up to 10 strong sources. Include source title, organization/author, date when available, URL, source type, and the exact limited point it can support. Search for dream science involving this concept, documented historical interpretations, and reliable biographical accounts when applicable. Do not rely on generic commercial dream dictionaries."
    response=client.responses.create(model=model,tools=[{'type':'web_search'}],instructions=WEB_SCOUT_INSTRUCTIONS,input=prompt)
    citations=[]
    try:
        payload=response.model_dump()
        for output in payload.get('output',[]):
            for content in output.get('content',[]) or []:
                for ann in content.get('annotations',[]) or []:
                    url=ann.get('url') or (ann.get('url_citation') or {}).get('url')
                    title=ann.get('title') or (ann.get('url_citation') or {}).get('title','')
                    if url: citations.append({'title':title,'url':url})
    except Exception: pass
    seen=set(); citations=[c for c in citations if not (c['url'] in seen or seen.add(c['url']))]
    return response.output_text,citations

@retrying(max_attempts=3, base_delay=1.0)
def structured_response(model: str, instructions: str, prompt: str, schema_model: type[BaseModel]) -> BaseModel:
    schema=schema_model.model_json_schema()
    response=client.responses.create(
        model=model,instructions=instructions,input=prompt,
        text={'format':{'type':'json_schema','name':schema_model.__name__.lower(),'strict':True,'schema':schema}}
    )
    return schema_model.model_validate_json(response.output_text)

## 8. Source normalization, evidence extraction, and deterministic QA

In [ ]:
class SourceCandidate(BaseModel):
    title: str
    url: str
    source_type: Literal['peer_reviewed','book','primary_historical','museum_archive','government','university','religious_text','reference','news_interview','other'] = 'other'
    publisher_or_journal: str = ''
    authors: list[str] = Field(default_factory=list)
    year: Optional[int] = None
    doi: Optional[str] = None
    pmid: Optional[str] = None
    abstract_or_excerpt: str = ''
    tradition_or_context: str = ''
    reliability_notes: str = ''
    quality_hint: float = Field(default=0.5, ge=0, le=1)

class SourceCandidateList(BaseModel):
    sources: list[SourceCandidate]

class DraftRecord(BaseModel):
    plain_definition: str
    common_contexts: list[str]
    interpretations: list[InterpretationPerspective]
    evidence_claims: list[EvidenceClaim]
    overall_summary: str
    themes: list[str]
    reflection_prompts: list[str]
    related_terms: list[str]
    evidence_label: str
    confidence_score: float
    safety_notes: list[str]
    cultural_context_notes: list[str]

class ModelQA(BaseModel):
    accepted: bool
    score: float = Field(ge=0, le=1)
    source_score: float = Field(ge=0, le=1)
    provenance_score: float = Field(ge=0, le=1)
    cultural_context_score: float = Field(ge=0, le=1)
    safety_score: float = Field(ge=0, le=1)
    clarity_score: float = Field(ge=0, le=1)
    problems: list[str]
    required_changes: list[str]

SCIENCE_CATEGORIES={'dream_experiences','body_health','emotions_states'}

def normalize_candidates(raw_candidates: list[dict], max_sources: int) -> list[SourceRecord]:
    output=[]; seen=set()
    for item in raw_candidates:
        url=(item.get('url') or '').strip(); title=(item.get('title') or '').strip()
        if not url or not title: continue
        normalized=url.split('#')[0]
        if normalized in seen: continue
        seen.add(normalized)
        verified,final_url=verify_url(url)
        quality=float(item.get('quality_hint',domain_weight(final_url)))
        output.append(SourceRecord(source_id=source_id(final_url,title),title=title,url=final_url,source_type=item.get('source_type','other'),publisher_or_journal=item.get('publisher_or_journal',''),authors=[a for a in item.get('authors',[]) if a],year=item.get('year'),doi=item.get('doi'),pmid=item.get('pmid'),accessed_at=now_iso(),abstract_or_excerpt=(item.get('abstract_or_excerpt') or '')[:7000],tradition_or_context=item.get('tradition_or_context',''),reliability_notes=item.get('reliability_notes',''),url_verified=verified,quality_score=max(0,min(1,quality))))
    output.sort(key=lambda s:(s.url_verified,s.quality_score),reverse=True)
    return output[:max_sources]

def deterministic_qa(record: DIDRecord, sources: list[SourceRecord], minimum_sources: int) -> tuple[float,list[str]]:
    problems=[]
    source_map={source.source_id:source for source in sources}
    cited=set(record.source_ids)
    for perspective in record.interpretations:
        cited.update(perspective.source_ids)
    for claim in record.evidence_claims:
        cited.update(claim.source_ids)
    missing=[source_id for source_id in cited if source_id not in source_map]
    if missing:
        problems.append(f'Unknown source IDs: {missing[:5]}')

    source_required = any(p.evidence_strength != 'reflective' for p in record.interpretations) or any(c.claim_type != 'reflective_synthesis' for c in record.evidence_claims)
    verified_count=sum(source.url_verified for source in sources)
    if source_required and verified_count < minimum_sources:
        problems.append(f'Only {verified_count} verified sources for source-derived claims; minimum is {minimum_sources}.')
    if not source_required and record.confidence_score > 0.5:
        problems.append('A reflective-only record cannot have confidence above 0.50.')

    text=' '.join([record.overall_summary,*[p.summary for p in record.interpretations],*[c.claim for c in record.evidence_claims]]).lower()
    banned=['this dream means you will','predicts your death','proves that','recovered memory is true','you have a disorder','definitely means']
    for phrase in banned:
        if phrase in text:
            problems.append(f'Unsafe certainty: {phrase}')
    for perspective in record.interpretations:
        if perspective.evidence_strength != 'reflective' and not perspective.source_ids:
            problems.append(f'Perspective lacks sources: {perspective.perspective}')

    source_score = min(1,verified_count/max(minimum_sources,1)) if source_required else 1.0
    provenance_score = 0 if missing else (min(1,len(cited)/max(1,len(record.interpretations))) if source_required else 1.0)
    safety_score = 0.4 if any('Unsafe certainty' in problem for problem in problems) else 1.0
    clarity_score = 1.0 if 30 <= len(record.overall_summary) <= 900 and len(record.reflection_prompts) >= 3 else 0.65
    score=0.35*source_score+0.3*provenance_score+0.2*safety_score+0.15*clarity_score
    return round(score,4),problems


## 9. End-to-end research function

In [ ]:
def checkpoint_path(term_id: str) -> Path: return CHECKPOINT_DIR/f'{term_id}.json'
def load_checkpoint(term_id: str) -> Optional[dict]:
    path=checkpoint_path(term_id)
    if path.exists():
        try:return json.loads(path.read_text())
        except Exception:return None
    return None

def save_checkpoint(term_id: str, payload: dict):
    temp=checkpoint_path(term_id).with_suffix('.tmp')
    temp.write_text(json.dumps(payload,indent=2,ensure_ascii=False))
    temp.replace(checkpoint_path(term_id))

def research_queries(term: str, category: str) -> list[str]:
    queries=[f'dreaming {term} dream research',f'{term} dreams psychology sleep']
    if category in SCIENCE_CATEGORIES: queries.append(f'{term} REM sleep dream study')
    return queries

def process_term(row: dict, settings: dict) -> dict:
    term_id=row['term_id']
    if settings['resume']:
        cached=load_checkpoint(term_id)
        if cached and cached.get('status') in {'accepted','rejected'}: return cached
    raw=[]; web_report=''; web_citations=[]
    try:
        for query in research_queries(row['term'],row['category']):
            if settings['use_pubmed']: raw.extend(search_pubmed(query,limit=5)); time.sleep(0.36 if not NCBI_API_KEY else 0.12)
            if settings['use_openalex']: raw.extend(search_openalex(query,limit=5))
            if settings['use_crossref']: raw.extend(search_crossref(query,limit=5))
        if settings['use_web']:
            web_report,web_citations=web_scout(row['term'],row['category'],settings['research_model'])
            # Let the structured extractor turn web citations/report into normalized source objects.
            extraction_prompt=f"""Term: {row['term']}
Category: {row['category']}
Web research report:
{web_report}
Citation annotations:
{json.dumps(web_citations,ensure_ascii=False)}

Return only real sources visible in the report or annotations. Verify that each URL/title pairing is plausible. Set url_verified false; local code will test it. Use conservative quality scores."""
            extracted=structured_response(settings['research_model'],SYNTHESIS_INSTRUCTIONS,extraction_prompt,SourceCandidateList)
            raw.extend([s.model_dump() for s in extracted.sources])
        sources=normalize_candidates(raw,settings['max_sources'])
        source_packet=json.dumps([s.model_dump() for s in sources],ensure_ascii=False)
        draft_prompt=f"""Build a DID record for:
TERM: {row['term']}
DISPLAY NAME: {row['display_name']}
CATEGORY: {row['category']}
ALIASES: {row['aliases']}
CATEGORY DESCRIPTION: {CATEGORY_DESCRIPTIONS[row['category']]}

VERIFIED SOURCE PACKET:
{source_packet}

WEB REPORT FOR CONTEXT ONLY:
{web_report[:10000]}

Requirements: Give plain-language possibilities, not one meaning. Include scientific material only when sources directly support it. Historical and cultural perspectives must be specific. Reflective synthesis may be included with evidence_strength='reflective' and no claim of scientific proof. If the record is reflective-only, set evidence_label to 'Reflective synthesis' and confidence_score no higher than 0.50. Use at least three useful personal questions. Overall summary 1–3 sentences."""
        draft=structured_response(settings['synthesis_model'],SYNTHESIS_INSTRUCTIONS,draft_prompt,DraftRecord)
        record=DIDRecord(record_id=term_id,term=row['term'],display_name=row['display_name'],category=row['category'],aliases=row['aliases'],reviewed_at=now_iso(),source_ids=sorted(set(draft.source_ids if hasattr(draft,'source_ids') else [sid for p in draft.interpretations for sid in p.source_ids]+[sid for c in draft.evidence_claims for sid in c.source_ids])),**draft.model_dump())
        det_score,det_problems=deterministic_qa(record,sources,settings['min_sources'])
        qa_prompt=f"""Review this DID record against the source packet and policy.
RECORD:
{record.model_dump_json(indent=2)}
SOURCES:
{source_packet}
Deterministic problems: {det_problems}
Return a strict QA result."""
        model_qa=structured_response(settings['research_model'],QA_INSTRUCTIONS,qa_prompt,ModelQA)
        combined=round(0.55*det_score+0.45*model_qa.score,4)
        accepted=combined>=settings['min_qa_score'] and model_qa.accepted and not det_problems
        qa=QAResult(record_id=term_id,accepted=accepted,score=combined,source_score=model_qa.source_score,provenance_score=model_qa.provenance_score,cultural_context_score=model_qa.cultural_context_score,safety_score=model_qa.safety_score,clarity_score=model_qa.clarity_score,problems=det_problems+model_qa.problems,required_changes=model_qa.required_changes)
        result={'status':'accepted' if accepted else 'rejected','record':record.model_dump(),'sources':[s.model_dump() for s in sources],'qa':qa.model_dump()}
    except Exception as exc:
        result={'status':'error','term':row,'error':f'{type(exc).__name__}: {exc}'}
    save_checkpoint(term_id,result)
    return result

## 10. Interactive build controller

The build writes a checkpoint after every concept. It can be stopped and resumed without losing accepted records.

In [ ]:
progress=widgets.IntProgress(value=0,min=0,max=1,description='DID build:')
status_html=widgets.HTML(value='<b>Ready.</b>')
log_output=widgets.Output(layout={'border':'1px solid #bbb','max_height':'320px','overflow':'auto'})
start_button=widgets.Button(description='Build Production DID',button_style='success',icon='play')
stop_button=widgets.Button(description='Stop after current record',button_style='warning',icon='stop')
stop_event=threading.Event()

def selected_rows(settings):
    rows=TAXONOMY_ROWS.copy()
    if settings['max_records']>0: rows=rows[:settings['max_records']]
    return rows

def run_build(_=None):
    stop_event.clear(); settings=resolved_settings(); rows=selected_rows(settings)
    progress.max=len(rows); progress.value=0; start_button.disabled=True
    accepted=rejected=errors=0
    with log_output:
        clear_output(); print(json.dumps(settings,indent=2)); print(f'Building {len(rows)} concepts...')
    # Conservative sequential controller. Increase CONCURRENCY only after validating API limits.
    for index,row in enumerate(rows,1):
        if stop_event.is_set(): break
        status_html.value=f'<b>{index}/{len(rows)}</b> — {row["display_name"]}'
        result=process_term(row,settings)
        progress.value=index
        if result['status']=='accepted': accepted+=1
        elif result['status']=='rejected': rejected+=1
        else: errors+=1
        with log_output: print(f'{index:4d} {result["status"]:8s} {row["display_name"]}')
    status_html.value=f'<b>Stopped/finished.</b> Accepted {accepted}, rejected {rejected}, errors {errors}. Run the export cell next.'
    start_button.disabled=False

def request_stop(_): stop_event.set(); status_html.value='<b>Stop requested.</b> The current record will finish first.'
build_thread=None
def start_in_background(_):
    global build_thread
    if build_thread and build_thread.is_alive(): return
    build_thread=threading.Thread(target=run_build,daemon=True)
    build_thread.start()
start_button.on_click(start_in_background); stop_button.on_click(request_stop)
display(widgets.HBox([start_button,stop_button]),progress,status_html,log_output)

### Optional code-driven run

The button above is the normal interactive workflow. This function provides the same behavior for unattended Colab execution.

In [ ]:
def run_full_build():
    run_build()
# Uncomment for unattended execution:
# run_full_build()

## 11. Compile, deduplicate, relate, and export

In [ ]:
def load_all_results():
    results=[]
    for path in sorted(CHECKPOINT_DIR.glob('*.json')):
        try: results.append(json.loads(path.read_text()))
        except Exception: pass
    return results

def merge_sources(results):
    merged={}
    for result in results:
        for source in result.get('sources',[]):
            sid=source['source_id']
            if sid not in merged or source.get('quality_score',0)>merged[sid].get('quality_score',0): merged[sid]=source
    return list(merged.values())

def build_relationships(records):
    edges=[]; by_theme=defaultdict(list)
    for record in records:
        for theme in record.get('themes',[]): by_theme[theme.lower()].append(record['record_id'])
        for related in record.get('related_terms',[]):
            target=slugify(related)
            edges.append({'source_id':record['record_id'],'target_id':target,'relationship':'explicit_related_term','weight':0.9})
    for theme,ids in by_theme.items():
        ids=list(dict.fromkeys(ids))[:100]
        for i,source in enumerate(ids):
            for target in ids[i+1:i+8]: edges.append({'source_id':source,'target_id':target,'relationship':f'shared_theme:{theme}','weight':0.55})
    # Deduplicate graph edges.
    seen=set(); output=[]
    for edge in edges:
        key=(edge['source_id'],edge['target_id'],edge['relationship'])
        if edge['source_id']!=edge['target_id'] and key not in seen: seen.add(key); output.append(edge)
    return output

def compact_for_app(record):
    return {'id':record['record_id'],'term':record['term'],'display_name':record['display_name'],'category':record['category'],'aliases':record.get('aliases',[]),'summary':record['overall_summary'],'themes':record.get('themes',[]),'prompts':record.get('reflection_prompts',[]),'confidence':record.get('confidence_score',0),'source_count':len(record.get('source_ids',[])),'evidence_label':record.get('evidence_label',''),'interpretations':[{'perspective':p['perspective'],'summary':p['summary'],'evidence_strength':p['evidence_strength'],'source_ids':p.get('source_ids',[])} for p in record.get('interpretations',[])],'safety_notes':record.get('safety_notes',[])}

def export_sqlite(records,sources,relationships,qa_rows,path):
    conn=sqlite3.connect(path)
    pd.json_normalize(records).to_sql('records',conn,if_exists='replace',index=False)
    pd.json_normalize(sources).to_sql('sources',conn,if_exists='replace',index=False)
    pd.DataFrame(relationships).to_sql('relationships',conn,if_exists='replace',index=False)
    pd.json_normalize(qa_rows).to_sql('qa',conn,if_exists='replace',index=False)
    conn.close()

def compile_and_export():
    results=load_all_results(); accepted=[r['record'] for r in results if r.get('status')=='accepted']; rejected=[r for r in results if r.get('status') in {'rejected','error'}]; sources=merge_sources(results); qa=[r['qa'] for r in results if 'qa' in r]; relationships=build_relationships(accepted)
    taxonomy={'version':'1.0','generated_at':now_iso(),'categories':CATEGORY_DESCRIPTIONS,'terms':TAXONOMY_ROWS}
    manifest={'database':'Somneiros Dream Interpretation Database','version':'1.0','generated_at':now_iso(),'run_started_at':RUN_STARTED_AT,'taxonomy_terms':len(TAXONOMY_ROWS),'accepted_records':len(accepted),'rejected_or_error':len(rejected),'sources':len(sources),'relationships':len(relationships),'evidence_policy':'Science, historical tradition, psychological theory, cultural context, and reflective synthesis are kept distinct.'}
    files={
        'did_records.json':accepted,'did_sources.json':sources,'did_relationships.json':relationships,
        'did_taxonomy.json':taxonomy,'did_app.json':[compact_for_app(r) for r in accepted],
        'did_rejected.json':rejected,'did_build_manifest.json':manifest
    }
    for name,data in files.items(): (OUTPUT_DIR/name).write_text(json.dumps(data,indent=2,ensure_ascii=False))
    pd.json_normalize(accepted).to_csv(OUTPUT_DIR/'did_records.csv',index=False)
    pd.json_normalize(sources).to_csv(OUTPUT_DIR/'did_sources.csv',index=False)
    pd.json_normalize(qa).to_csv(OUTPUT_DIR/'did_qa_report.csv',index=False)
    export_sqlite(accepted,sources,relationships,qa,OUTPUT_DIR/'somneiros_did.sqlite')
    bundle=OUTPUT_DIR/'somneiros_did_bundle.zip'
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in OUTPUT_DIR.iterdir():
            if path.is_file() and path!=bundle: archive.write(path,path.name)
    print(json.dumps(manifest,indent=2)); print('Bundle:',bundle)
    return manifest

export_button=widgets.Button(description='Compile and Export DID',button_style='info',icon='download')
export_output=widgets.Output()
def export_click(_):
    with export_output: clear_output(); compile_and_export()
export_button.on_click(export_click); display(export_button,export_output)

## 12. Interactive DID inspector

Use this after export to search accepted records, inspect interpretations, and review their source provenance before publishing them to the PWA.

In [ ]:
search_box=widgets.Text(placeholder="Search water, being chased, house, teeth...",description="Search:")
category_box=widgets.Dropdown(options=["All"]+sorted(SEED_TAXONOMY),value="All",description="Category:")
results_box=widgets.Output()

def inspect_database(*_):
    path=OUTPUT_DIR/"did_records.json"
    with results_box:
        clear_output()
        if not path.exists():
            print("Compile the database first.")
            return
        records=json.loads(path.read_text())
        query=search_box.value.lower().strip()
        category=category_box.value
        matches=[r for r in records if (category=="All" or r["category"]==category) and (not query or query in json.dumps(r,ensure_ascii=False).lower())][:20]
        print(f"{len(matches)} result(s) shown")
        for record in matches:
            questions="\n".join("- "+p for p in record["reflection_prompts"])
            card=(
                f"### {record['display_name']}\n"
                f"**Category:** {record['category']}  \n"
                f"**Evidence:** {record['evidence_label']} ({record['confidence_score']:.2f})  \n\n"
                f"{record['overall_summary']}\n\n"
                f"**Themes:** {', '.join(record['themes'])}\n\n"
                f"**Questions:**\n{questions}"
            )
            display(Markdown(card))
search_box.observe(inspect_database,names="value")
category_box.observe(inspect_database,names="value")
display(widgets.HBox([search_box,category_box]),results_box)

## 13. Publish to the Somneiros PWA

After the build passes review:

1. Download `somneiros_did_bundle.zip`.
2. Copy `did_app.json` into the repository’s `data/` folder.
3. Optionally archive the larger research files under `data/research/` or a separate data repository.
4. Commit and redeploy GitHub Pages.
5. Confirm the app reports the expected record and category counts under **Dream Interpretation Database**.

Do not publish rejected records. Keep checkpoint and rejected files for editorial review.